# Chapitre 2 — Leçon 4 : Pipelines scikit-learn

## Objectifs d'apprentissage

À la fin de cette leçon, vous serez capable de :
- **Expliquer** pourquoi les pipelines préviennent le data leakage
- **Construire** un Pipeline simple avec preprocessing et modèle
- **Utiliser** ColumnTransformer pour traiter différemment colonnes numériques et catégorielles
- **Créer** un pipeline complet production-ready

---

## 🎯 Accroche : Le cauchemar du code spaghetti

Imaginez ce scénario : vous avez écrit un modèle ML qui fonctionne parfaitement. Six mois plus tard, vous devez le réutiliser. Vous ouvrez votre code et vous trouvez :

```python
# Quelque part dans le notebook...
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# ... 50 cellules plus loin ...
encoder = OneHotEncoder()
X_train_encoded = encoder.fit_transform(X_train_cat)

# ... encore plus loin ...
X_train_final = np.concatenate([X_train_scaled, X_train_encoded], axis=1)

# ... et le modèle ...
model.fit(X_train_final, y_train)
```

**Questions :**
- Dans quel ordre appliquer les transformations ?
- Avez-vous sauvegardé tous les transformers ?
- Comment reproduire exactement la même séquence en production ?

*(Réponse attendue : C'est un cauchemar — code fragile, difficile à maintenir, et risque élevé de data leakage)*

---

## 4.1 Pourquoi les Pipelines ?

### Trois problèmes, une solution

```
┌─────────────────────────────────────────────────────────────────┐
│                  PROBLÈMES SANS PIPELINE                        │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  ❌ PROBLÈME 1 : Data Leakage                                   │
│     Facile d'oublier d'appliquer .fit() seulement sur train    │
│                                                                 │
│  ❌ PROBLÈME 2 : Reproductibilité                               │
│     Difficile de reproduire exactement les mêmes étapes        │
│                                                                 │
│  ❌ PROBLÈME 3 : Déploiement                                    │
│     Impossible de sauvegarder "la séquence complète"           │
│                                                                 │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  ✅ SOLUTION : Pipeline                                         │
│     Un objet unique qui encapsule TOUTE la séquence            │
│     - .fit() s'applique à TOUS les composants sur train        │
│     - .predict() applique les transformations + prédiction     │
│     - Sauvegardable en un seul fichier                         │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Importations
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

print("✅ Imports chargés")

### Créons un dataset réaliste

Pour illustrer les pipelines, créons un dataset avec :
- Des colonnes **numériques** (qui nécessitent du scaling)
- Des colonnes **catégorielles** (qui nécessitent de l'encoding)
- Des **valeurs manquantes** (à imputer)

In [ ]:
# Créer un dataset réaliste avec différents types de données
np.random.seed(42)
n = 200

data = pd.DataFrame({
    'age': np.random.randint(18, 70, n),
    'salaire': np.random.randint(20000, 150000, n),
    'anciennete': np.random.randint(0, 30, n),
    'departement': np.random.choice(['IT', 'RH', 'Finance', 'Marketing'], n),
    'niveau_etude': np.random.choice(['Bac', 'Licence', 'Master', 'PhD'], n),
})

# Ajouter des valeurs manquantes (réaliste !)
data.loc[np.random.choice(n, 10), 'age'] = np.nan
data.loc[np.random.choice(n, 15), 'departement'] = np.nan

# Target : promotion (oui/non)
data['promotion'] = ((data['anciennete'] > 5) & (data['salaire'] < 80000)).astype(int)
# Ajouter du bruit
noise_idx = np.random.choice(n, 30)
data.loc[noise_idx, 'promotion'] = 1 - data.loc[noise_idx, 'promotion']

print("Dataset créé :")
print(data.head(10))
print(f"\nValeurs manquantes :")
print(data.isnull().sum())

In [ ]:
# Séparer features et target
X = data.drop('promotion', axis=1)
y = data['promotion']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train : {len(X_train)} exemples")
print(f"Test : {len(X_test)} exemples")

---

## 4.2 Pipeline Simple

Un `Pipeline` est une liste d'étapes nommées, exécutées dans l'ordre.

```
┌─────────────────────────────────────────────────────────────────┐
│                    STRUCTURE D'UN PIPELINE                      │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│   Pipeline([                                                    │
│       ('nom_etape_1', Transformer1()),                          │
│       ('nom_etape_2', Transformer2()),                          │
│       ('nom_etape_3', Model())           ← Dernier = estimator  │
│   ])                                                            │
│                                                                 │
│   Quand vous appelez .fit() :                                   │
│   1. etape_1.fit_transform(X)                                   │
│   2. etape_2.fit_transform(X_transformé)                        │
│   3. etape_3.fit(X_final, y)                                    │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### Exemple minimaliste : Scaler + Modèle

Commençons simple : uniquement les colonnes numériques.

In [ ]:
# Sélectionner seulement les colonnes numériques pour cet exemple
colonnes_num = ['age', 'salaire', 'anciennete']
X_train_num = X_train[colonnes_num].copy()
X_test_num = X_test[colonnes_num].copy()

# Imputer les valeurs manquantes AVANT le pipeline (pour cet exemple simple)
X_train_num = X_train_num.fillna(X_train_num.median())
X_test_num = X_test_num.fillna(X_train_num.median())  # Utilise médiane du train !

print("Données numériques prêtes :")
print(X_train_num.head())

In [ ]:
# Pipeline simple : Scaler → Modèle
pipeline_simple = Pipeline([
    ('scaler', StandardScaler()),           # Étape 1 : normaliser
    ('classifier', LogisticRegression())    # Étape 2 : classifier
])

print("Pipeline créé :")
print(pipeline_simple)

In [ ]:
# Entraîner le pipeline ENTIER en une seule ligne !
pipeline_simple.fit(X_train_num, y_train)

print("✅ Pipeline entraîné !")
print(f"\nCe que le scaler a appris :")
print(f"  Moyennes : {pipeline_simple.named_steps['scaler'].mean_}")

In [ ]:
# Prédire et évaluer (le pipeline applique AUTOMATIQUEMENT le scaling)
score = pipeline_simple.score(X_test_num, y_test)

print(f"Accuracy sur test : {score:.2%}")

**Question :** Quand vous appelez `pipeline.score(X_test, y_test)`, que se passe-t-il en interne ?

*(Réponse attendue : Le pipeline applique d'abord scaler.transform(X_test), puis classifier.predict() sur le résultat, et compare avec y_test)*

<details>
<summary>🤔 Question Socratique : Pourquoi le pipeline utilise-t-il .transform() et non .fit_transform() sur le test set ?</summary>

### 🔑 Réponse

C'est la magie du Pipeline ! Il **sait** automatiquement :

- Pendant `.fit()` → appeler `.fit_transform()` sur chaque étape (sauf la dernière)
- Pendant `.predict()` ou `.score()` → appeler seulement `.transform()`

Vous n'avez **jamais** à vous soucier de data leakage quand vous utilisez un Pipeline correctement. Il gère tout pour vous !

```python
# Ce que le pipeline fait en interne lors de .fit() :
X_scaled = scaler.fit_transform(X_train)  # fit + transform
classifier.fit(X_scaled, y_train)

# Ce que le pipeline fait lors de .predict() :
X_scaled = scaler.transform(X_new)  # SEULEMENT transform
predictions = classifier.predict(X_scaled)
```

</details>

---

## 4.3 ColumnTransformer : Traitement différencié par type

Dans un dataset réel, vous avez des colonnes de types différents :
- **Numériques** → Imputation + Scaling
- **Catégorielles** → Imputation + Encoding

`ColumnTransformer` permet d'appliquer des transformations **différentes** à des colonnes **différentes**.

```
┌─────────────────────────────────────────────────────────────────┐
│                    COLUMNTRANSFORMER                            │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│                      ┌──────────────────────┐                   │
│   age, salaire ─────►│ Imputer + Scaler     │────┐              │
│   (numériques)       └──────────────────────┘    │              │
│                                                  ├──► X_final   │
│                      ┌──────────────────────┐    │              │
│   departement,  ────►│ Imputer + Encoder    │────┘              │
│   niveau_etude       └──────────────────────┘                   │
│   (catégorielles)                                               │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Identifier les colonnes par type
colonnes_numeriques = ['age', 'salaire', 'anciennete']
colonnes_categorielles = ['departement', 'niveau_etude']

print(f"Colonnes numériques : {colonnes_numeriques}")
print(f"Colonnes catégorielles : {colonnes_categorielles}")

In [ ]:
# Pipeline pour colonnes numériques
pipeline_numerique = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),  # Remplacer NaN par médiane
    ('scaler', StandardScaler())                     # Normaliser
])

# Pipeline pour colonnes catégorielles
pipeline_categoriel = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),  # Remplacer NaN par mode
    ('encoder', OneHotEncoder(handle_unknown='ignore'))     # One-hot encoding
])

print("✅ Pipelines par type créés")

In [ ]:
# Combiner avec ColumnTransformer
preprocesseur = ColumnTransformer([
    ('num', pipeline_numerique, colonnes_numeriques),
    ('cat', pipeline_categoriel, colonnes_categorielles)
])

print("ColumnTransformer créé :")
print(preprocesseur)

In [ ]:
# Tester le preprocesseur seul
X_train_preprocessed = preprocesseur.fit_transform(X_train)

print(f"Avant preprocessing : {X_train.shape}")
print(f"Après preprocessing : {X_train_preprocessed.shape}")
print(f"\n(Plus de colonnes car OneHotEncoder crée une colonne par catégorie)")

**Question :** Pourquoi le nombre de colonnes a-t-il augmenté après preprocessing ?

*(Réponse attendue : Le OneHotEncoder transforme chaque valeur catégorielle en plusieurs colonnes binaires — une par catégorie)*

---

## 4.4 Pipeline Complet : Preprocessing + Modèle

Maintenant, combinons **tout** : ColumnTransformer + Modèle dans un seul Pipeline.

In [ ]:
# PIPELINE COMPLET PRODUCTION-READY
pipeline_complet = Pipeline([
    ('preprocesseur', preprocesseur),              # Étape 1 : tout le preprocessing
    ('classifier', RandomForestClassifier(        # Étape 2 : le modèle
        n_estimators=100,
        max_depth=5,
        random_state=42
    ))
])

print("Pipeline complet :")
print(pipeline_complet)

In [ ]:
# ENTRAÎNER tout en une ligne
pipeline_complet.fit(X_train, y_train)

print("✅ Pipeline complet entraîné !")

In [ ]:
# ÉVALUER
score_train = pipeline_complet.score(X_train, y_train)
score_test = pipeline_complet.score(X_test, y_test)

print(f"Performance du pipeline complet :")
print(f"  - Accuracy train : {score_train:.2%}")
print(f"  - Accuracy test  : {score_test:.2%}")

# Vérifier l'overfitting
if score_train - score_test < 0.1:
    print(f"\n✅ Pas d'overfitting majeur détecté")
else:
    print(f"\n⚠️ Possible overfitting")

In [ ]:
# PRÉDIRE sur de nouvelles données
nouvelles_personnes = pd.DataFrame({
    'age': [35, 28, 55],
    'salaire': [45000, 120000, 60000],
    'anciennete': [8, 2, 15],
    'departement': ['IT', 'Finance', 'RH'],
    'niveau_etude': ['Master', 'PhD', 'Licence']
})

predictions = pipeline_complet.predict(nouvelles_personnes)
probas = pipeline_complet.predict_proba(nouvelles_personnes)

print("Prédictions pour nouvelles personnes :")
print("─" * 60)
for i, row in nouvelles_personnes.iterrows():
    promo = "OUI" if predictions[i] == 1 else "NON"
    proba = probas[i][1]
    print(f"{row['departement']:10} | Age {row['age']} | {row['anciennete']} ans | Promotion: {promo} ({proba:.1%})")

### 📖 Définition

```
┌─────────────────────────────────────────────────────────────────┐
│ 📖 DÉFINITION : Pipeline scikit-learn                           │
│                                                                 │
│ Un Pipeline est une séquence ordonnée de transformateurs        │
│ et d'un estimateur final, encapsulée dans un seul objet.        │
│ Il garantit que :                                               │
│                                                                 │
│ • Toutes les transformations sont appliquées dans l'ordre       │
│ • .fit() n'est appelé que sur les données d'entraînement        │
│ • Le workflow est reproductible et sauvegardable                │
│ • Le data leakage est automatiquement évité                     │
│                                                                 │
│ C'est la bonne pratique standard pour tout projet ML.           │
└─────────────────────────────────────────────────────────────────┘
```

<details>
<summary>🤔 Question Socratique : Quels sont les 3 avantages principaux d'utiliser un Pipeline plutôt que du code séparé ?</summary>

### 🔑 Réponse

**1. Prévention du data leakage**
- Le pipeline garantit que `.fit()` n'est appelé que sur les données d'entraînement
- Impossible d'accidentellement `fit_transform()` sur le test set

**2. Reproductibilité**
- L'ordre des opérations est fixé et documenté
- Pas de risque d'oublier une étape
- Le même pipeline peut être réappliqué sur de nouvelles données

**3. Déploiement simplifié**
- Un seul objet à sauvegarder (`joblib.dump(pipeline, 'model.pkl')`)
- En production, charger et appeler `.predict()` — c'est tout !
- Pas besoin de recréer manuellement chaque étape

</details>

---

## 🎯 Template réutilisable

Voici un template que vous pouvez copier-coller pour vos projets :

In [ ]:
# =====================================================
# TEMPLATE PIPELINE PRODUCTION-READY
# =====================================================

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier  # ou autre modèle

def creer_pipeline(colonnes_num, colonnes_cat, modele):
    """
    Crée un pipeline complet avec preprocessing et modèle.
    
    Args:
        colonnes_num: liste des colonnes numériques
        colonnes_cat: liste des colonnes catégorielles
        modele: estimateur scikit-learn
    
    Returns:
        Pipeline prêt à l'emploi
    """
    # Pipeline numérique
    num_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    
    # Pipeline catégoriel
    cat_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ])
    
    # Combiner
    preprocesseur = ColumnTransformer([
        ('num', num_pipeline, colonnes_num),
        ('cat', cat_pipeline, colonnes_cat)
    ])
    
    # Pipeline complet
    pipeline = Pipeline([
        ('preprocesseur', preprocesseur),
        ('modele', modele)
    ])
    
    return pipeline

# Utilisation
mon_pipeline = creer_pipeline(
    colonnes_num=['age', 'salaire', 'anciennete'],
    colonnes_cat=['departement', 'niveau_etude'],
    modele=RandomForestClassifier(n_estimators=100, random_state=42)
)

print("Template prêt !")

---

## 🔄 Accéder aux composants du Pipeline

Parfois, vous avez besoin d'inspecter un composant spécifique :

In [ ]:
# Accéder aux étapes par nom
print("Noms des étapes :")
for nom, etape in pipeline_complet.named_steps.items():
    print(f"  - {nom}: {type(etape).__name__}")

In [ ]:
# Accéder au modèle pour voir les feature importances
modele = pipeline_complet.named_steps['classifier']
importances = modele.feature_importances_

print(f"\nFeature importances (top 5) :")
indices = np.argsort(importances)[::-1][:5]
for i in indices:
    print(f"  Feature {i}: {importances[i]:.4f}")

---

## 🧠 Réflexion métacognitive

1. **Pourquoi le Pipeline est-il considéré comme une bonne pratique "production-ready" ?**

2. **Pouvez-vous expliquer à un collègue** comment ColumnTransformer gère les colonnes de types différents ?

3. **Dans votre prochain projet**, utiliserez-vous un Pipeline ou du code séparé ? Pourquoi ?

---

## 📝 Résumé

| Concept | Description |
|---------|-------------|
| `Pipeline` | Séquence d'étapes exécutées dans l'ordre |
| `ColumnTransformer` | Applique différentes transformations à différentes colonnes |
| Avantage #1 | Prévient automatiquement le data leakage |
| Avantage #2 | Workflow reproductible et documenté |
| Avantage #3 | Sauvegardable/chargeable en un seul fichier |
| `.named_steps` | Accéder aux composants individuels |

**Pattern à retenir :**
```python
pipeline = Pipeline([
    ('preprocesseur', ColumnTransformer([...])),
    ('modele', MonModele())
])
pipeline.fit(X_train, y_train)
pipeline.score(X_test, y_test)
```

---

## ➡️ Prochaine leçon

Dans la **Leçon 2.5 : Premier modèle complet**, nous allons mettre tout ensemble : charger un vrai dataset, construire un pipeline, entraîner, évaluer, et **comparer avec une baseline** — le test ultime pour savoir si votre modèle est vraiment utile.

**Question de transition :** Si votre modèle a 85% d'accuracy, comment savez-vous si c'est "bon" ou "mauvais" ?